In [98]:
import pandas as pd
import numpy as np
import math

### Task 1

In [99]:
df= pd.read_csv("insurance.csv")

In [100]:
df.head(10)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


In [101]:
df.shape
# 1338 rows and 7 columns
# column labels represent information about an individual
# Label is likely charges
# The rest of the columns minus index and charges

(1338, 7)

### Task 2: Make a Guess

In [102]:
# player1: $10k
# player2: $5k

### Task 3: Compute a Mean Baseline

In [103]:
df["mean_baseline"] = df["charges"].mean()
mean_baseline = df["charges"].mean()
df.head()

,age,sex,bmi,children,smoker,region,charges,mean_baseline
0,19,female,27.900,0,yes,southwest,16884.92400,13270.422265
1,18,male,33.770,1,no,southeast,1725.55230,13270.422265
2,28,male,33.000,3,no,southeast,4449.46200,13270.422265
3,33,male,22.705,0,no,northwest,21984.47061,13270.422265
4,32,male,28.880,0,no,northwest,3866.85520,13270.422265


-  Q1: Player1 was almost spot on.
-  Q1: player 2 was double the guess.

-  Q2: No, because ther are some big outliers that bring the mean up.


### Task 4: Compute MAE for Your Baseline

In [104]:
df["charges"].mean()
df["errors"] = abs(df["charges"] - mean_baseline)
baseline_mae = df["errors"].mean()

print(baseline_mae)


9091.12658113703


-  Q1: If MAE is 4,000, what does that mean?
-  A: The averages of all the charges are $4,000 off

### Task 5: Compute MSE and RMSE

In [105]:
 
df["errors_squared"] = df["errors"] * df["errors"]
baseline_mse = df["errors_squared"].mean()
print(baseline_mse)

baseline_rmse = math.sqrt(baseline_mse)
print(baseline_rmse)

146542766.49354792
12105.484975561612


-  $s is the unit used
-  RMSE is preferred when you want to penalize big errors

#### Task 6: Build a Grouped Baseline

In [106]:
df["smoker_mean"] = df.groupby("smoker")["charges"].transform("mean")
df.head()
smoker_mean = df.groupby("smoker")["charges"].mean()
print(smoker_mean)



smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64


In [107]:
df["error_v2"] = df["charges"] - df["smoker_mean"]

df["errors_v2"] = abs(df["charges"] - df["smoker_mean"])
v2_baseline_mae = df["errors_v2"].mean()

print(f" baseline v2: {v2_baseline_mae}")
print(f" baseline v1: {baseline_mae}")


 baseline v2: 5662.08960934306
 baseline v1: 9091.12658113703


-  is grouped mae(v2) lower or higher than baseline (v1): Lower
-  What does this say about specificity: the more specifics entered increases the model's potential to be more accurate 

### Task 7: Label and Seaparate Features

In [108]:
x = df[["age", "sex", "bmi", "children", "smoker", "region"]]
y = df["charges"]

print(x.head(3))
print(y.head(3))

#Split x and y instead of keeping them together because we are trying to predict y (charges) in our model.

   age     sex    bmi  children smoker     region
0   19  female  27.90         0    yes  southwest
1   18    male  33.77         1     no  southeast
2   28    male  33.00         3     no  southeast
0    16884.9240
1     1725.5523
2     4449.4620
Name: charges, dtype: float64


### Task 8: Simulate a Train/Test Split

In [109]:
y.shape

(1338,)

In [110]:
split=round(len(y) * .8)

y_train = pd.DataFrame(y.head(split))
x_train = pd.DataFrame(x.head(split))

y_test = pd.DataFrame(y.tail(len(y)-split))
x_test = pd.DataFrame(x.tail(len(x)-split))

print(len(y_train))  
print(len(y_test))
print(len(x_train))
print(len(x_test))


1070
268
1070
268


-  1.Why is it a problem to test a model on the same data it trained on? 
-  A: It will learn the data and may not predict the way we want it to.

-  2.What is this problem called?
-  A:Overfitting 

### Task 9: Evaluate on Test Data Only

In [111]:
y_train.head()

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520


In [112]:
y_train_mean = y_train["charges"].mean()

y_test["errors"] = abs(y_test["charges"] - y_train_mean)
test_baseline_mae = y_test["errors"].mean()

print(test_baseline_mae)


9197.532234648968


In [113]:
y_train["smoker"] = x_train["smoker"]
y_test["smoker"] = x_test["smoker"]


smoker_averages = y_train.groupby("smoker")["charges"].mean()
y_train.head()

y_test["smoker_mean"] = y_test["smoker"].map(smoker_averages)

#y_smoker_mean = df.groupby("smoker")["charges"].mean()
print(smoker_averages)

smoker
no      8496.645098
yes    31974.339819
Name: charges, dtype: float64


In [ ]:
y_test["errors_v2"] = y_test["charges"] - y_test["smoker_mean"]

y_test["errors_v2"] = abs(y_test["charges"] - y_test["smoker_mean"])
test_grouped_mae = y_test["errors_v2"].mean()

print(f" grouped smoker mae: {test_grouped_mae}")
print(f" test baseline: {test_baseline_mae}")

print(f" baseline v2: {v2_baseline_mae}")
print(f" baseline v1: {baseline_mae}")

# baselines were slightly higher for smoker grouped, but lower for the ungrouped. It tells us the model is
# slightly better for smoker charges, but worse for predicting ungrouped.  

 grouped smoker mae: 5504.588416638439
 test baseline: 9197.532234648968
 baseline v2: 5662.08960934306
 baseline v1: 9091.12658113703


In [116]:
df.head()

,age,sex,bmi,children,smoker,region,charges,mean_baseline,errors,errors_squared,smoker_mean,error_v2,errors_v2
0,19,female,27.900,0,yes,southwest,16884.92400,13270.422265,3614.501735,1.306462e+07,32050.231832,-15165.307832,15165.307832
1,18,male,33.770,1,no,southeast,1725.55230,13270.422265,11544.869965,1.332840e+08,8434.268298,-6708.715998,6708.715998
2,28,male,33.000,3,no,southeast,4449.46200,13270.422265,8820.960265,7.780934e+07,8434.268298,-3984.806298,3984.806298
3,33,male,22.705,0,no,northwest,21984.47061,13270.422265,8714.048345,7.593464e+07,8434.268298,13550.202312,13550.202312
4,32,male,28.880,0,no,northwest,3866.85520,13270.422265,9403.567065,8.842707e+07,8434.268298,-4567.413098,4567.413098


### Task 10: Summarize Findings

The dataset was general information on customers the model is looking to predict charges. Player1 was very close, player 2 was way off. If we used our intuitive guesses, there would be more error than the mean baseline. The same goes for grouped baseline vs mean baseline. MAE & RMSE are similar with RMSE punishing more for error margins. MSE punishes the most for margin errors but is harder to relate back to the information since the numbers are so high. Use MAE and RMSE when you want the same units as your data and MSE for comparing raw squared error values.

### Advanced Tasks 
Second grouping variable

In [133]:
df = pd.read_csv("insurance.csv")

In [138]:

split=round(len(df) * .8)

train = df.head(split)

test = df.tail(len(df)-split)



In [140]:
smoker_region = train.groupby(["smoker", "region"])["charges"].mean()
smoker_region = pd.DataFrame(smoker_region)
smoker_region

charges
smoker region                 
no     northeast   9160.079717
       northwest   8720.769672
       southeast   8104.174761
       southwest   8042.259303
yes    northeast  30826.921381
       northwest  29094.628881
       southeast  34779.706825
       southwest  31776.463818

In [ ]:
x_train = train[["age", "sex", "bmi", "children", "smoker", "region"]]
y_train = pd.DataFrame(train["charges"])

x_test = test[["age", "sex", "bmi", "children", "smoker", "region"]]
y_test = pd.DataFrame(test["charges"])

In [142]:
y_test["smoker"] = x_test["smoker"]
y_test["region"] = x_test["region"]
y_test["smoker_region"] = y_test[["smoker","region"]].map(smoker_region)

TypeError: the first argument must be callable